# TranSTR CausalVid - Training Notebook
# Notebook này train model TranSTR cho CausalVidQA
# Weights và checkpoints được lưu trên W&B

In [ ]:
import os
# --- Git Clone & Setup ---
REPO_URL = "https://github.com/DanielQH07/tranSTR_Casual.git"
REPO_NAME = "tranSTR_Casual"
BRANCH = "origin"

if not os.path.exists(REPO_NAME):
    print(f"Cloning {REPO_URL}...")
    !git clone {REPO_URL} -b {BRANCH}
else:
    print("Repo already exists.")

# Change Directory to the repo root
if os.path.basename(os.getcwd()) != REPO_NAME:
    try:
        target_dir = os.path.join(os.getcwd(), REPO_NAME, "causalvid")
        if os.path.exists(target_dir):
             os.chdir(target_dir)
        elif os.path.exists(REPO_NAME):
             os.chdir(REPO_NAME)

        print(f"Changed directory to: {os.getcwd()}")
    except Exception as e:
             print(f"Could not set working directory: {e}")

In [ ]:
# CELL 2: Install & Login W&B
print('=== CELL 2: W&B Setup ===')
%pip install -q wandb --upgrade
import wandb
import os

# ============================================
# WANDB CONFIG
# ============================================
WANDB_API_KEY = ''  # 🔴 Paste your W&B API key here
WANDB_PROJECT = 'transtr-Origin-causalvid'
WANDB_ENTITY = None

wandb.login(key=WANDB_API_KEY)
print('✅ W&B logged in successfully!')

In [ ]:
# CELL 3: Patch DataLoader.py (Fast - split-first flow)
DATALOADER_CODE = '''
import torch
import os
import json
import pandas as pd
import pickle as pkl
import os.path as osp
import numpy as np
from torch.utils.data import Dataset
from utils.util import transform_bb

class VideoQADataset(Dataset):
    def __init__(self, split, n_query=5, obj_num=20, sample_list_path="/anno",
                 video_feature_path="/vit", object_feature_path="/obj", split_dir=None,
                 topK_frame=16, max_samples=None, verbose=True):
        super().__init__()
        self.split = split
        self.mc = n_query
        self.obj_num = obj_num
        self.video_feature_path = video_feature_path
        self.object_feature_path = object_feature_path
        self.topK_frame = topK_frame
        split_alias = "valid" if split == "val" else split

        # 1. Detect object feature format
        self.obj_format = "flat"
        if osp.exists(object_feature_path):
            for item in os.listdir(object_feature_path)[:5]:
                if "features_node" in item or "part" in item:
                    self.obj_format = "kaggle_subdirs"
                    break
        if verbose:
            print(f"[{split}] Object feature format: {self.obj_format}")

        # 2. Build object feature index (vid -> pkl path) - one pass
        self.obj_feature_map = {}
        if self.obj_format == "kaggle_subdirs":
            for sub in os.listdir(object_feature_path):
                sub_path = osp.join(object_feature_path, sub)
                if osp.isdir(sub_path):
                    for fname in os.listdir(sub_path):
                        if fname.endswith(".pkl"):
                            self.obj_feature_map[fname[:-4]] = osp.join(sub_path, fname)
        else:
            for fname in os.listdir(object_feature_path):
                if fname.endswith(".pkl"):
                    self.obj_feature_map[fname[:-4]] = osp.join(object_feature_path, fname)

        # 3. Load split video IDs (pkl > txt)
        valid_vids = []
        if split_dir:
            pkl_path = osp.join(split_dir, f"{split_alias}.pkl")
            txt_path = osp.join(split_dir, f"{split_alias}.txt")
            if osp.exists(pkl_path):
                with open(pkl_path, "rb") as f:
                    data = pkl.load(f)
                if isinstance(data, (list, tuple, set, np.ndarray)):
                    valid_vids = [str(v) for v in data]
                elif isinstance(data, dict):
                    valid_vids = [str(k) for k in data.keys()]
                if verbose:
                    print(f"[{split}] Loaded {len(valid_vids)} video IDs from {pkl_path}")
            elif osp.exists(txt_path):
                with open(txt_path, encoding="utf-8") as f:
                    valid_vids = [l.strip() for l in f if l.strip()]
                if verbose:
                    print(f"[{split}] Loaded {len(valid_vids)} video IDs from {txt_path}")

        # 4. Optional: limit
        if max_samples is not None and len(valid_vids) > max_samples:
            valid_vids = valid_vids[:max_samples]
            if verbose:
                print(f"[{split}] Limited to {max_samples} videos")

        # 5. Filter by ViT + OBJ availability (per-vid check, not full scan)
        valid_set = []
        n_vit = 0
        n_obj = 0
        for vid in valid_vids:
            has_vit = osp.exists(osp.join(video_feature_path, f"{vid}.pt"))
            has_obj = vid in self.obj_feature_map
            n_vit += has_vit
            n_obj += has_obj
            if has_vit and has_obj:
                valid_set.append(vid)
        if verbose:
            print(f"[{split}] ViT: {n_vit}, Obj: {n_obj}, Both: {len(valid_set)}")

        # 6. Load annotations only for valid videos
        data_rows = []
        for vid in valid_set:
            t_json = osp.join(sample_list_path, vid, "text.json")
            a_json = osp.join(sample_list_path, vid, "answer.json")
            if not (osp.exists(t_json) and osp.exists(a_json)):
                continue
            try:
                with open(t_json, encoding="utf-8") as f:
                    t = json.load(f)
                with open(a_json, encoding="utf-8") as f:
                    a = json.load(f)
                for key in ["descriptive", "explanatory", "predictive", "counterfactual"]:
                    if key in t and key in a:
                        q, ans = t[key], a[key]
                        if "question" in q and "answer" in q and "answer" in ans:
                            row = {"video_id": vid, "question": q["question"], "answer": ans["answer"], "type": key}
                            for i, c in enumerate(q["answer"]):
                                row[f"a{i}"] = c
                            data_rows.append(row)
                        if key in ["predictive", "counterfactual"] and "reason" in q and "reason" in ans:
                            row = {"video_id": vid, "question": "Why?", "answer": ans["reason"], "type": f"{key}_reason"}
                            for i, c in enumerate(q["reason"]):
                                row[f"a{i}"] = c
                            data_rows.append(row)
            except Exception:
                pass

        self.sample_list = pd.DataFrame(data_rows)
        if verbose:
            print(f"[{split}] Final: {len(self.sample_list)} QA pairs")

    def __len__(self):
        return len(self.sample_list)

    def __getitem__(self, idx):
        cur = self.sample_list.iloc[idx]
        vid = str(cur["video_id"])
        qns = str(cur["question"])
        ans_id = int(cur["answer"])
        ans_word = [f"{qns} [SEP] {cur[f'a{i}']}" for i in range(self.mc)]
        qns_key = f"{vid}_{cur['type']}"

        frame_feat = torch.load(osp.join(self.video_feature_path, f"{vid}.pt"), weights_only=True)
        if isinstance(frame_feat, np.ndarray):
            frame_feat = torch.from_numpy(frame_feat)
        frame_feat = frame_feat.float()

        nf = frame_feat.shape[0]
        if nf > self.topK_frame:
            idx_f = np.linspace(0, nf - 1, self.topK_frame).astype(int)
            frame_feat = frame_feat[idx_f]
        elif nf < self.topK_frame:
            frame_feat = torch.cat([frame_feat, torch.zeros(self.topK_frame - nf, frame_feat.shape[1])], 0)

        obj_feat = self._load_obj(vid)
        return frame_feat, obj_feat, qns, ans_word, ans_id, qns_key

    def _load_obj(self, vid):
        objs = []
        pkl_path = self.obj_feature_map.get(vid)
        if pkl_path:
            try:
                with open(pkl_path, "rb") as f:
                    data = pkl.load(f)
                feats = np.array(data.get("features", data.get("feat", [])))
                bboxes = np.array(data.get("bboxes", data.get("bbox", [])))
                nf = feats.shape[0] if len(feats.shape) > 1 else 0
                indices = np.linspace(0, nf - 1, self.topK_frame).astype(int) if nf > self.topK_frame else range(nf)
                for i in indices:
                    feat = torch.from_numpy(feats[i]).float()
                    bbox = torch.from_numpy(bboxes[i]).float()
                    if feat.shape[0] > self.obj_num:
                        feat, bbox = feat[:self.obj_num], bbox[:self.obj_num]
                    elif feat.shape[0] < self.obj_num:
                        p = self.obj_num - feat.shape[0]
                        feat = torch.cat([feat, torch.zeros(p, feat.shape[1])], 0)
                        bbox = torch.cat([bbox, torch.zeros(p, bbox.shape[1])], 0)
                    bb = torch.from_numpy(transform_bb(bbox.numpy(), 640, 480)).float()
                    objs.append(torch.cat([feat, bb], -1))
            except Exception:
                pass
        while len(objs) < self.topK_frame:
            objs.append(torch.zeros(self.obj_num, 2053))
        return torch.stack(objs)
'''
with open('DataLoader.py', 'w') as f:
    f.write(DATALOADER_CODE)
print('✅ DataLoader.py patched (split-first flow + max_samples + verbose)')

In [ ]:
# CELL 4: Imports (with module reload)
import os, torch, numpy as np, pandas as pd
from torch.utils.data import DataLoader
from utils.util import set_seed, set_gpu_devices
from networks.model import VideoQAmodel
import torch.nn as nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
import wandb
import sys
import importlib

# Force reload DataLoader (in case cell 3 was re-run after first import)
if 'DataLoader' in sys.modules:
    importlib.reload(sys.modules['DataLoader'])
from DataLoader import VideoQADataset

print('✅ Imports OK (DataLoader fresh reload)')
print(f'   VideoQADataset signature includes: max_samples={"max_samples" in VideoQADataset.__init__.__code__.co_varnames}')

In [ ]:
# CELL 5: Train/Eval Functions (with Gradient Accumulation)
def train(model, optimizer, loader, xe, device, use_amp=True, scaler=None, accum_steps=1):
    """Train with gradient accumulation. Effective bs = loader.batch_size * accum_steps"""
    model.train()
    total_loss, preds, gts = 0, [], []
    optimizer.zero_grad()
    
    for step, batch in enumerate(loader):
        f, o, q, a, ans_id, _ = batch
        f, o, tgt = f.to(device), o.to(device), ans_id.to(device)
        
        with torch.amp.autocast('cuda', enabled=use_amp):
            out = model(f, o, q, a)
            loss = xe(out, tgt) / accum_steps
        
        if scaler:
            scaler.scale(loss).backward()
        else:
            loss.backward()
        
        if (step + 1) % accum_steps == 0 or (step + 1) == len(loader):
            if scaler:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            optimizer.zero_grad()
        
        total_loss += loss.item() * accum_steps
        preds.append(out.argmax(-1))
        gts.append(ans_id)
    
    preds = torch.cat(preds).cpu()
    gts = torch.cat(gts)
    return total_loss / len(loader), (preds == gts).float().mean().item() * 100

def eval_model(model, loader, device):
    model.eval()
    preds, gts = [], []
    with torch.no_grad():
        for batch in loader:
            f, o, q, a, ans_id, _ = batch
            out = model(f.to(device), o.to(device), q, a)
            preds.append(out.argmax(-1))
            gts.append(ans_id)
    preds = torch.cat(preds).cpu()
    gts = torch.cat(gts)
    return (preds == gts).float().mean().item() * 100

print('✅ Functions defined (gradient accumulation)')

In [ ]:
# CELL 6: Data Paths (Kaggle vs Colab)
RUN_TRAINING = True

import os
from pathlib import Path

BASE = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content/working"
MODEL_DIR = os.path.join(BASE, "models")
os.makedirs(MODEL_DIR, exist_ok=True)

# Detect environment
IS_KAGGLE = os.path.exists("/kaggle/input")
IS_COLAB = os.path.exists("/content") and not IS_KAGGLE

print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Colab' if IS_COLAB else 'Local'}")

if IS_KAGGLE:
    # ========== KAGGLE: Datasets already mounted ==========
    # Add your datasets in Kaggle notebook settings
    VIDEO_FEATURE_ROOT = "/kaggle/input/vit-features-full-merged"
    OBJ_DIR = "/kaggle/input/object-detection-causal-full"
    ANNOTATION_QA_PATH = "/kaggle/input/text-annotation/QA"
    SPLIT_DIR_PATH = "/kaggle/input/casual-vid-data-split/split"
    
    # Auto-detect actual paths
    def _find_dir_with_ext(root, ext):
        root = Path(root)
        if not root.exists():
            return str(root)
        for p in root.rglob(f'*{ext}'):
            return str(p.parent)
        return str(root)
    
    def _find_dir_containing(root, name):
        root = Path(root)
        if not root.exists():
            return str(root)
        for p in root.rglob('*'):
            if p.is_dir() and p.name.lower() == name.lower():
                return str(p)
        return str(root)
    
    VIDEO_FEATURE_ROOT = _find_dir_with_ext("/kaggle/input/vit-features-full-merged", '.pt')
    ANNOTATION_QA_PATH = _find_dir_containing("/kaggle/input/text-annotation", 'QA')
    SPLIT_DIR_PATH = _find_dir_containing("/kaggle/input/casual-vid-data-split", 'split')
    
    # Find OBJ_DIR with features_node_X_FULL folders
    obj_base = "/kaggle/input/object-detection-causal-full"
    if os.path.exists(obj_base):
        for item in os.listdir(obj_base):
            if 'features_node' in item:
                OBJ_DIR = obj_base
                break
            subpath = os.path.join(obj_base, item)
            if os.path.isdir(subpath):
                for sub in os.listdir(subpath):
                    if 'features_node' in sub:
                        OBJ_DIR = subpath
                        break

else:
    # ========== COLAB: Download via kagglehub ==========
    import subprocess, sys
    
    try:
        from google.colab import userdata
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
        print('✅ Kaggle credentials loaded')
    except Exception:
        pass
    
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub', '--upgrade'], check=False)
    import kagglehub
    
    print('Downloading Kaggle datasets...')
    VIT_ROOT = kagglehub.dataset_download("danielq07/vit-features-full-merged")
    OBJ_ROOT = kagglehub.dataset_download("danielq07/object-detection-causal-full")
    ANNO_ROOT = kagglehub.dataset_download("lusnaw/text-annotation")
    SPLIT_ROOT = kagglehub.dataset_download("danielq07/casual-vid-data-split")
    
    def _find_dir_with_ext(root, ext):
        root = Path(root)
        if not root.exists():
            return str(root)
        for p in root.rglob(f'*{ext}'):
            return str(p.parent)
        return str(root)
    
    def _find_dir_containing(root, name):
        root = Path(root)
        if not root.exists():
            return str(root)
        for p in root.rglob('*'):
            if p.is_dir() and p.name.lower() == name.lower():
                return str(p)
        return str(root)
    
    VIDEO_FEATURE_ROOT = _find_dir_with_ext(VIT_ROOT, '.pt')
    ANNOTATION_QA_PATH = _find_dir_containing(ANNO_ROOT, 'QA')
    SPLIT_DIR_PATH = _find_dir_containing(SPLIT_ROOT, 'split')
    
    OBJ_DIR = OBJ_ROOT
    for item in os.listdir(OBJ_ROOT):
        if 'features_node' in item:
            OBJ_DIR = OBJ_ROOT
            break
        subpath = os.path.join(OBJ_ROOT, item)
        if os.path.isdir(subpath):
            for sub in os.listdir(subpath):
                if 'features_node' in sub:
                    OBJ_DIR = subpath
                    break

print(f'✅ VIDEO_FEATURE_ROOT: {VIDEO_FEATURE_ROOT}')
print(f'✅ OBJ_DIR: {OBJ_DIR}')
print(f'✅ ANNOTATION_QA_PATH: {ANNOTATION_QA_PATH}')
print(f'✅ SPLIT_DIR_PATH: {SPLIT_DIR_PATH}')

In [ ]:
# CELL 7: Config
class Config:
    video_feature_root = globals().get('VIDEO_FEATURE_ROOT', "/kaggle/input/vit-features-full-merged")
    object_feature_path = globals().get('OBJ_DIR', "/kaggle/input/object-detection-causal-full")
    sample_list_path = globals().get('ANNOTATION_QA_PATH', "/kaggle/input/text-annotation/QA")
    split_dir_txt = globals().get('SPLIT_DIR_PATH', "/kaggle/input/casual-vid-data-split/split")

    topK_frame = 16
    frame_feat_dim = 1024
    obj_feat_dim = 2053
    objs = 20

    bs = 8
    accum_steps = 4  # effective = 32

    lr = 1e-4
    epoch = 15
    gpu = 0
    dropout = 0.3
    encoder_dropout = 0.3
    patience = 5
    gamma = 0.1
    decay = 1e-4
    n_query = 5
    d_model = 512
    word_dim = 768
    topK_obj = 10
    num_encoder_layers = 2
    num_decoder_layers = 2
    nheads = 8
    normalize_before = True
    activation = 'gelu'
    
    # Text encoder
    text_encoder_lr = 5e-6
    freeze_text_encoder = False
    text_encoder_type = 'microsoft/deberta-base'  # Better than bert-base-uncased
    text_pool_mode = 1
    
    # DeBERTa có vấn đề với FP16 → tắt AMP
    use_amp = False  # Disable for DeBERTa stability
    
    hard_eval = False
    pos_ratio = 0.7
    neg_ratio = 0.3
    a = 1.0
    num_workers = 2

args = Config()
set_gpu_devices(args.gpu)
set_seed(999)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

effective_bs = args.bs * args.accum_steps
print(f'Device: {device}')
print(f'Batch: {args.bs} x {args.accum_steps} = {effective_bs} effective')
print(f'Text encoder: {args.text_encoder_type}')
print(f'AMP: {args.use_amp} (disabled for DeBERTa)')

In [ ]:
# CELL 8: Create Datasets (force reload DataLoader)
import sys, importlib

# Force reload DataLoader to pick up latest cell 3 patch
for mod in list(sys.modules.keys()):
    if mod == 'DataLoader' or mod.startswith('DataLoader.'):
        del sys.modules[mod]
import DataLoader as _DL
importlib.reload(_DL)
from DataLoader import VideoQADataset

# Verify signature
sig_vars = VideoQADataset.__init__.__code__.co_varnames
assert 'max_samples' in sig_vars, f"❌ DataLoader.py chưa được patch! Re-run cell 3 trước.\nSignature hiện tại: {sig_vars}"
print(f'✅ VideoQADataset.__init__ has: max_samples, verbose')

print('\n=== CELL 8: Datasets ===')
MAX_TRAIN_SAMPLES = None  # Set int (e.g. 2000) to limit train videos
PIN_MEMORY = torch.cuda.is_available()

print('\n--- Creating TRAIN dataset ---')
train_ds = VideoQADataset(
    split='train', n_query=args.n_query, obj_num=args.objs,
    sample_list_path=args.sample_list_path, video_feature_path=args.video_feature_root,
    object_feature_path=args.object_feature_path, split_dir=args.split_dir_txt,
    topK_frame=args.topK_frame, max_samples=MAX_TRAIN_SAMPLES, verbose=True
)

print('\n--- Creating VAL dataset ---')
val_ds = VideoQADataset(
    split='val', n_query=args.n_query, obj_num=args.objs,
    sample_list_path=args.sample_list_path, video_feature_path=args.video_feature_root,
    object_feature_path=args.object_feature_path, split_dir=args.split_dir_txt,
    topK_frame=args.topK_frame, max_samples=None, verbose=True
)

print('\n--- Creating TEST dataset ---')
test_ds = VideoQADataset(
    split='test', n_query=args.n_query, obj_num=args.objs,
    sample_list_path=args.sample_list_path, video_feature_path=args.video_feature_root,
    object_feature_path=args.object_feature_path, split_dir=args.split_dir_txt,
    topK_frame=args.topK_frame, max_samples=None, verbose=True
)

train_loader = DataLoader(train_ds, args.bs, shuffle=True, num_workers=args.num_workers, pin_memory=PIN_MEMORY)
val_loader = DataLoader(val_ds, args.bs, shuffle=False, num_workers=args.num_workers, pin_memory=PIN_MEMORY)
test_loader = DataLoader(test_ds, args.bs, shuffle=False, num_workers=args.num_workers, pin_memory=PIN_MEMORY)

print('\n' + '='*60)
print('DATASET SUMMARY')
print('='*60)
print(f'Train: {len(train_ds)} samples -> {len(train_loader)} batches')
print(f'Val:   {len(val_ds)} samples -> {len(val_loader)} batches')
print(f'Test:  {len(test_ds)} samples -> {len(test_loader)} batches')
print('='*60)

if len(train_ds) > 0:
    print('\nSanity check: Loading first batch...')
    try:
        ff, of, qns, ans, ans_id, keys = next(iter(train_loader))
        print(f'  ViT features: {ff.shape}')
        print(f'  Object features: {of.shape}')
        print(f'  Answer IDs: {ans_id[:8]}')
        print('✅ Sanity check PASSED!')
    except Exception as e:
        print(f'  ❌ ERROR: {e}')
        import traceback; traceback.print_exc()

In [ ]:
# CELL 9: Model + W&B Init
cfg = {k: v for k, v in Config.__dict__.items() if not k.startswith('_')}
cfg['device'] = device
model = VideoQAmodel(**cfg)
optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.decay)
scheduler = ReduceLROnPlateau(optimizer, 'max', factor=args.gamma, patience=args.patience)
model.to(device)
xe = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler('cuda', enabled=args.use_amp)
print(f'Model: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params')

# W&B Init (uses WANDB_PROJECT, WANDB_ENTITY from cell 2)
wandb_config = {k: v for k, v in Config.__dict__.items() if not k.startswith('_') and not callable(v)}
wandb_config['device'] = str(device)
wandb_config['effective_bs'] = args.bs * args.accum_steps

wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    config=wandb_config,
    name=f'transtr-bs{args.bs}x{args.accum_steps}-ep{args.epoch}',
    save_code=True
)
print(f'✅ W&B run: {wandb.run.name}')

In [ ]:
# CELL 10: Training Loop
BEST_MODEL_FILENAME = 'best_model.ckpt'
LAST_CKPT_FILENAME = 'last_checkpoint.ckpt'
save_path = os.path.join(MODEL_DIR, BEST_MODEL_FILENAME)
last_ckpt_path = os.path.join(MODEL_DIR, LAST_CKPT_FILENAME)

RESUME_FROM_LAST_CKPT = True
best_acc = 0.0
best_epoch = 0
start_epoch = 1

if RESUME_FROM_LAST_CKPT and os.path.exists(last_ckpt_path):
    try:
        ckpt = torch.load(last_ckpt_path, map_location=device)
        model.load_state_dict(ckpt['model_state_dict'], strict=False)
        if 'optimizer_state_dict' in ckpt:
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        if 'scheduler_state_dict' in ckpt:
            scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        if scaler and 'scaler_state_dict' in ckpt and ckpt['scaler_state_dict']:
            scaler.load_state_dict(ckpt['scaler_state_dict'])
        best_acc = float(ckpt.get('best_acc', 0.0))
        best_epoch = int(ckpt.get('best_epoch', 0))
        start_epoch = int(ckpt.get('epoch', 0)) + 1
        print(f"✅ Resumed: epoch={start_epoch-1}, best={best_acc:.2f}%")
        wandb.log({'resume/start_epoch': start_epoch, 'resume/best_acc': best_acc})
    except Exception as e:
        print(f"⚠️ Resume failed: {e}")

if RUN_TRAINING:
    effective_bs = args.bs * args.accum_steps
    if start_epoch > args.epoch:
        print(f'Already at epoch {args.epoch}. Skip.')
    else:
        print(f'Training: bs={args.bs} x accum={args.accum_steps} = {effective_bs}')
        for ep in range(start_epoch, args.epoch + 1):
            loss, acc = train(model, optimizer, train_loader, xe, device, args.use_amp, scaler, args.accum_steps)
            val_acc = eval_model(model, val_loader, device)
            scheduler.step(val_acc)
            print(f'Ep {ep}: Loss={loss:.4f}, Train={acc:.1f}%, Val={val_acc:.1f}%')

            ckpt = {
                'epoch': ep,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'scaler_state_dict': scaler.state_dict() if scaler else None,
                'best_acc': float(max(best_acc, val_acc)),
                'best_epoch': int(best_epoch if val_acc <= best_acc else ep),
                'config': {'bs': args.bs, 'accum_steps': args.accum_steps}
            }
            torch.save(ckpt, last_ckpt_path)

            wandb.log({'epoch': ep, 'train_loss': loss, 'train_acc': acc, 'val_acc': val_acc,
                       'lr': optimizer.param_groups[0]['lr'], 'best_acc_so_far': max(best_acc, val_acc)})

            last_art = wandb.Artifact('last-checkpoint', type='checkpoint', metadata={'epoch': ep})
            last_art.add_file(last_ckpt_path, name=LAST_CKPT_FILENAME)
            wandb.log_artifact(last_art, aliases=['latest', f'epoch-{ep}'])

            if val_acc > best_acc:
                best_acc = val_acc
                best_epoch = ep
                torch.save(model.state_dict(), save_path)
                ckpt['best_acc'] = float(best_acc)
                ckpt['best_epoch'] = int(best_epoch)
                torch.save(ckpt, last_ckpt_path)
                print(f'  ✅ New best: {val_acc:.2f}%')

                best_art = wandb.Artifact('best-model', type='model', metadata={'epoch': ep, 'val_acc': float(val_acc)})
                best_art.add_file(save_path, name=BEST_MODEL_FILENAME)
                wandb.log_artifact(best_art, aliases=['best', f'epoch-{ep}'])

        print(f'\n✅ Done! Best: {best_acc:.2f}% @ epoch {best_epoch}')

    wandb.run.summary['best_val_acc'] = float(best_acc)
    wandb.run.summary['best_epoch'] = int(best_epoch)
else:
    print('Skip training (RUN_TRAINING=False)')

In [ ]:
# CELL 11: TEST Evaluation (Full Metrics + Acc_ALL)
import matplotlib.pyplot as plt
import json

if os.path.exists(save_path):
    model.load_state_dict(torch.load(save_path, map_location=device))
model.eval()

results = {}
type_map = {'descriptive':'D', 'explanatory':'E', 'predictive':'P', 
            'predictive_reason':'PR', 'counterfactual':'C', 'counterfactual_reason':'CR'}

with torch.no_grad():
    for batch in test_loader:
        f, o, q, a, ans_id, keys = batch
        out = model(f.to(device), o.to(device), q, a)
        preds = out.argmax(-1).cpu().numpy()
        for i, k in enumerate(keys):
            for ts, tsh in type_map.items():
                if k.endswith('_' + ts):
                    vid = k[:-(len(ts) + 1)]
                    if vid not in results:
                        results[vid] = {}
                    results[vid][tsh] = (preds[i] == ans_id[i].item())
                    break

stats = {k: [0, 0] for k in ['D', 'E', 'P', 'PR', 'C', 'CR', 'PAR', 'CAR']}
for vid, r in results.items():
    for t in ['D', 'E', 'P', 'PR', 'C', 'CR']:
        if t in r:
            stats[t][1] += 1
            stats[t][0] += int(r[t])
    if 'P' in r and 'PR' in r:
        stats['PAR'][1] += 1
        stats['PAR'][0] += int(r['P'] and r['PR'])
    if 'C' in r and 'CR' in r:
        stats['CAR'][1] += 1
        stats['CAR'][0] += int(r['C'] and r['CR'])

print('=' * 60)
print('EVALUATION RESULTS - TEST SET')
print('=' * 60)
metric_names = {'D': 'Description', 'E': 'Explanation', 'P': 'Predictive-Ans', 
                'PR': 'Predictive-Reason', 'C': 'Counterfactual-Ans', 'CR': 'Counterfactual-Reason',
                'PAR': 'PAR (P+PR)', 'CAR': 'CAR (C+CR)'}

eval_metrics = {}
for k in ['D', 'E', 'P', 'PR', 'C', 'CR']:
    c, t = stats[k]
    acc = c / t * 100 if t else 0
    eval_metrics[k] = acc
    print(f"{metric_names[k]:<25} ==>   {acc:.2f}%  ({c}/{t})")

print('-' * 60)
for k in ['PAR', 'CAR']:
    c, t = stats[k]
    acc = c / t * 100 if t else 0
    eval_metrics[k] = acc
    print(f"{metric_names[k]:<25} ==>   {acc:.2f}%  ({c}/{t} paired)")

acc_all = (eval_metrics['D'] + eval_metrics['E'] + eval_metrics['PAR'] + eval_metrics['CAR']) / 4
eval_metrics['Acc_ALL'] = acc_all
print('-' * 60)
print(f"{'Acc (ALL)':<25} ==>   {acc_all:.2f}%  ((D+E+PAR+CAR)/4)")
print('=' * 60)

wandb_metrics = {f'test/{k}': v for k, v in eval_metrics.items()}
wandb.log(wandb_metrics)
wandb.run.summary.update(wandb_metrics)

final_art = wandb.Artifact('final-results', type='results', 
                           metadata={'best_val_acc': float(best_acc), 'best_epoch': int(best_epoch)})
if os.path.exists(save_path):
    final_art.add_file(save_path, name=BEST_MODEL_FILENAME)
if os.path.exists(last_ckpt_path):
    final_art.add_file(last_ckpt_path, name=LAST_CKPT_FILENAME)

eval_json_path = os.path.join(MODEL_DIR, 'test_metrics.json')
with open(eval_json_path, 'w') as f:
    json.dump(eval_metrics, f, indent=2)
final_art.add_file(eval_json_path, name='test_metrics.json')
wandb.log_artifact(final_art)

labels = ['D', 'E', 'PAR', 'CAR', 'Acc_ALL']
accs = [eval_metrics[k] for k in labels]
plt.figure(figsize=(10, 5))
bars = plt.bar(labels, accs, color=['#4CAF50', '#2196F3', '#FF9800', '#E91E63', '#9C27B0'])
plt.ylim(0, 100)
plt.ylabel('Accuracy %')
plt.title('TranSTR Test Performance')
for bar in bars:
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{bar.get_height():.1f}%', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'test_results.png'))
plt.show()

wandb.finish()
print('✅ W&B run finished')